# YOLO11s 가구 탐지 모델 사용 방법

이 노트북에서는 학습된 YOLO11s 모델을 사용해 원룸 사진 속 가구를 탐지한다.

## 중요

이 모델은 `pt` 파일만 단독으로 사용하는 것이 아니라,  
클래스별 confidence threshold 후처리를 함께 적용해야 한다.

기본 `model.predict(conf=0.25)`처럼 하나의 threshold를 모든 클래스에 적용하면 성능이 달라진다.  
최종 사용 방식은 다음과 같다.

1. YOLO 모델로 낮은 confidence 기준(`conf=0.05`)에서 후보 박스를 넓게 검출한다.
2. 검출된 박스에 대해 클래스별 threshold를 따로 적용한다.
3. threshold를 통과한 박스만 최종 가구 후보로 사용한다.

## 최종 threshold

| class | threshold |
|---|---:|
| bed | 0.11 |
| couch | 0.35 |
| desk | 0.30 |
| refrigerator | 0.75 |
| wardrob / wardrobe | 0.37 |

## 최종 모델 경로

```python
/content/drive/MyDrive/DL 4조/yolo11s_train/yolo11s_train.pt

# threshold 설정

In [ ]:
from pathlib import Path
from ultralytics import YOLO
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
import cv2
import torch
import pandas as pd

# =========================
# 1. 경로 설정
# =========================

MODEL_PATH = Path("/content/drive/MyDrive/DL 4조/yolo11s_train/yolo11s_train.pt")

# 이미 이미지가 들어있는 폴더 경로로 수정
SOURCE_DIR = Path("/content/test_images")

# 결과 저장 폴더
SAVE_DIR = Path("/content/furniture_yolo_predictions")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

print("MODEL_PATH 존재:", MODEL_PATH.exists(), MODEL_PATH)
print("SOURCE_DIR 존재:", SOURCE_DIR.exists(), SOURCE_DIR)

# =========================
# 2. 모델 로드
# =========================

model = YOLO(str(MODEL_PATH))

DEVICE = 0 if torch.cuda.is_available() else "cpu"

print("모델 로드 완료")
print("DEVICE:", DEVICE)
print("model.names:", model.names)

# =========================
# 3. 클래스별 threshold 설정
# =========================

CLASS_THRESHOLDS = {
    "bed": 0.11,
    "couch": 0.35,
    "desk": 0.30,
    "refrigerator": 0.75,
    "wardrob": 0.37,
    "wardrobe": 0.37
}

print("CLASS_THRESHOLDS:", CLASS_THRESHOLDS)

# =========================
# 4. 후처리 예측
# =========================

results = model.predict(
    source=str(SOURCE_DIR),
    imgsz=832,
    conf=0.05,   # 먼저 낮게 후보를 넓게 뽑음
    iou=0.6,
    device=DEVICE,
    save=False,
    verbose=False
)

final_detections = []

for r in results:
    img_path = Path(r.path)

    img = cv2.imread(str(img_path))
    if img is None:
        print("이미지 읽기 실패:", img_path)
        continue

    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    if r.boxes is not None and len(r.boxes) > 0:
        boxes = r.boxes.xyxy.cpu().numpy()
        confs = r.boxes.conf.cpu().numpy()
        clss = r.boxes.cls.cpu().numpy().astype(int)

        for box, conf, cls_id in zip(boxes, confs, clss):
            class_name = model.names[int(cls_id)]

            threshold = CLASS_THRESHOLDS.get(class_name, 0.25)

            # 클래스별 threshold 적용
            if conf < threshold:
                continue

            x1, y1, x2, y2 = map(int, box)
            label = f"{class_name} {conf:.2f}"

            cv2.rectangle(
                img,
                (x1, y1),
                (x2, y2),
                (255, 0, 0),
                2
            )

            cv2.putText(
                img,
                label,
                (x1, max(y1 - 8, 20)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7,
                (255, 0, 0),
                2
            )

            final_detections.append({
                "image": img_path.name,
                "class_id": int(cls_id),
                "class_name": class_name,
                "conf": float(conf),
                "threshold_used": float(threshold),
                "x1": x1,
                "y1": y1,
                "x2": x2,
                "y2": y2
            })

    save_path = SAVE_DIR / img_path.name
    Image.fromarray(img).save(save_path)

print("후처리 예측 완료")
print("결과 이미지 저장 폴더:", SAVE_DIR)
print("최종 탐지 개수:", len(final_detections))

detections_df = pd.DataFrame(final_detections)
display(detections_df)